# Notebook 16 - Experiment 24: Fairness-Weighted Ensemble
### Novelty 7
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Every other novelty in this study is about *finding* bias. This one actually tries to
do something about it, which makes it the only experiment here that ends in a
recommendation rather than a diagnosis.

I build three soft-voting ensembles from the five baselines, changing only how the
votes are weighted:

- **Equal** — every model counts the same
- **Performance-weighted** — weighted by macro F1
- **Fairness-weighted** — weighted by the inverse of mean AOD, so the models with
  the smallest fairness gaps get the most say

What I'm really asking is whether the accuracy-best weights and the fairness-best
weights pull in the same direction or against each other. If the fairness-weighted
version cuts AOD without costing much accuracy, that's something you could actually
recommend deploying, not just an observation.

No training at all — every baseline already saved its probability array, so the
ensemble is just arithmetic on files I already have. Fills **Table 13**.

## Cell 1: Setup

In [ ]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
!pip install -q aif360 fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.0 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 7.8 MB/s eta 0:00:00


## Cell 2: Load Saved Probability Arrays

This is why every experiment saved its probabilities. Building an ensemble now
costs nothing.

In [ ]:
MODELS = [("exp01", "LogisticRegression"),
          ("exp02", "LinearSVM"),
          ("exp03", "MultinomialNB"),
          ("exp04", "RandomForest"),
          ("exp05", "XGBoost")]

preds, probas, weights_f1, weights_aod = {}, {}, {}, {}
from sklearn.metrics import f1_score

for exp_id, name in MODELS:
    try:
        p = load_predictions(exp_id, name)
    except FileNotFoundError:
        print(f"  missing: {name}"); continue

    preds[name] = p
    proba_cols  = sorted([c_ for c_ in p.columns if c_.startswith("proba_")])
    probas[name] = p[proba_cols].values

    f1 = f1_score(p["y_true"], p["y_pred"], average="macro")
    weights_f1[name] = f1

    audit = full_fairness_audit(p)
    aod = audit["AIF360 AOD"].abs().mean() if "AIF360 AOD" in audit else np.nan
    weights_aod[name] = aod

    print(f"  {name:<20}: macro F1 {f1:.4f}   mean |AOD| {aod:.4f}")

y_true    = preds[MODELS[0][1]]["y_true"].values
subgroups = preds[MODELS[0][1]]["subgroup"].values
classes   = sorted(pd.unique(y_true))
print(f"\nLoaded {len(probas)} models. Classes: {classes}")

pip install 'aif360[inFairness]'


  LogisticRegression  : macro F1 0.5672   mean |AOD| 0.0312
  LinearSVM           : macro F1 0.5648   mean |AOD| 0.0329
  MultinomialNB       : macro F1 0.5589   mean |AOD| 0.0061
  RandomForest        : macro F1 0.4169   mean |AOD| 0.0285
  XGBoost             : macro F1 0.4519   mean |AOD| 0.0336

Loaded 5 models. Classes: ['negative', 'neutral', 'positive']


## Cell 3: Build the Three Ensembles

Soft voting is a weighted average of probability arrays followed by argmax. The
only difference between the three variants is the weight vector.

In [ ]:
def soft_vote(prob_dict, weights):
    names = list(prob_dict.keys())
    w = np.array([weights[n] for n in names], dtype=float)
    w = w / w.sum()
    stacked = np.stack([prob_dict[n] for n in names])          # (models, n, classes)
    avg = np.tensordot(w, stacked, axes=(0, 0))                # (n, classes)
    idx = avg.argmax(axis=1)
    return np.array([classes[i] for i in idx]), avg

# 1. equal
w_equal = {n: 1.0 for n in probas}
pred_eq, proba_eq = soft_vote(probas, w_equal)

# 2. performance-weighted
pred_pf, proba_pf = soft_vote(probas, weights_f1)

# 3. fairness-weighted — inverse AOD, so fairer models weigh more
w_fair = {}
for n, aod in weights_aod.items():
    w_fair[n] = 1.0 / max(abs(aod), 1e-4) if not np.isnan(aod) else 1.0
pred_fw, proba_fw = soft_vote(probas, w_fair)

print("WEIGHTS")
print("="*70)
names = list(probas.keys())
wf = np.array([w_fair[n] for n in names]); wf = wf / wf.sum()
wp = np.array([weights_f1[n] for n in names]); wp = wp / wp.sum()
for i, n in enumerate(names):
    print(f"  {n:<20}: equal {1/len(names):.3f}   "
          f"performance {wp[i]:.3f}   fairness {wf[i]:.3f}")

print("\nThe fairness weights should favour whichever model had the")
print("smallest AOD, even if that model was not the most accurate.")

WEIGHTS
  LogisticRegression  : equal 0.200   performance 0.222   fairness 0.110
  LinearSVM           : equal 0.200   performance 0.221   fairness 0.104
  MultinomialNB       : equal 0.200   performance 0.218   fairness 0.563
  RandomForest        : equal 0.200   performance 0.163   fairness 0.120
  XGBoost             : equal 0.200   performance 0.177   fairness 0.102

The fairness weights should favour whichever model had the
smallest AOD, even if that model was not the most accurate.


## Cell 4: Evaluate and Save

In [ ]:
variants = [("Equal-weighted",       pred_eq, proba_eq),
            ("Performance-weighted", pred_pf, proba_pf),
            ("Fairness-weighted",    pred_fw, proba_fw)]

ens_results = []
for label, yp, pr in variants:
    save_predictions("exp24", label.replace(" ", "_"),
                     y_true, yp, pr, subgroups)
    res = evaluate_model(y_true, yp, pr, label=label)
    res["Variant"] = label
    ens_results.append(res)
    print(f"{label:<22}: macro F1 {res['Macro F1']:.4f}   "
          f"accuracy {res['Accuracy']:.4f}")

print("\nBest individual baseline for reference:")
best_single = max(weights_f1, key=weights_f1.get)
print(f"  {best_single}: macro F1 {weights_f1[best_single]:.4f}")

  saved predictions -> exp24_Equal-weighted_TweetEval.parquet  (12,284 rows)
Equal-weighted        : macro F1 0.5516   accuracy 0.5952
  saved predictions -> exp24_Performance-weighted_TweetEval.parquet  (12,284 rows)
Performance-weighted  : macro F1 0.5585   accuracy 0.5959
  saved predictions -> exp24_Fairness-weighted_TweetEval.parquet  (12,284 rows)
Fairness-weighted     : macro F1 0.5622   accuracy 0.5935

Best individual baseline for reference:
  LogisticRegression: macro F1 0.5672


## Cell 5: Fairness Comparison — the Actual Test

In [ ]:
rows = []
for label, _, _ in variants:
    p = load_predictions("exp24", label.replace(" ", "_"))
    audit = full_fairness_audit(p)
    row = {"Variant": label,
           "Macro F1": f1_score(p["y_true"], p["y_pred"], average="macro")}
    if "AIF360 AOD" in audit.columns:
        row["Mean AOD"] = audit["AIF360 AOD"].abs().mean()
        for _, a in audit.iterrows():
            row[f"{a['Subgroup']} AOD"] = a.get("AIF360 AOD", np.nan)
    wga, worst = worst_group_accuracy(p)
    row["WGA"] = wga
    row["Worst Subgroup"] = worst
    rows.append(row)

ens_fair = pd.DataFrame(rows)
print("="*90)
print("ENSEMBLE FAIRNESS COMPARISON")
print("="*90)
print(ens_fair.round(4).to_string(index=False))

if "Mean AOD" in ens_fair.columns:
    eq = ens_fair[ens_fair["Variant"] == "Equal-weighted"].iloc[0]
    fw = ens_fair[ens_fair["Variant"] == "Fairness-weighted"].iloc[0]
    d_aod = fw["Mean AOD"] - eq["Mean AOD"]
    d_f1  = fw["Macro F1"] - eq["Macro F1"]

    print(f"\nFairness-weighted vs equal-weighted:")
    print(f"  AOD change      : {d_aod:+.4f}")
    print(f"  macro F1 change : {d_f1:+.4f}")

    if d_aod < -0.01 and d_f1 > -0.02:
        print("\n  Fairness improved at negligible accuracy cost.")
        print("  This is a deployable recommendation.")
    elif d_aod < -0.01:
        print(f"\n  Fairness improved but accuracy fell by {abs(d_f1):.4f}.")
        print("  The trade-off is real and should be reported as such.")
    else:
        print("\n  Fairness weighting did not improve AOD here. That is still")
        print("  a finding: aggregation weighting is not a reliable remedy.")

ENSEMBLE FAIRNESS COMPARISON
             Variant  Macro F1  Mean AOD  emoji-heavy AOD  slang-heavy AOD  sarcasm AOD    WGA Worst Subgroup
      Equal-weighted    0.5516    0.0272          -0.0191           0.0156       0.0468 0.5848    emoji-heavy
Performance-weighted    0.5585    0.0307          -0.0160           0.0208       0.0552 0.5869         formal
   Fairness-weighted    0.5622    0.0123          -0.0168           0.0167      -0.0034 0.5851         formal

Fairness-weighted vs equal-weighted:
  AOD change      : -0.0149
  macro F1 change : +0.0106

  Fairness improved at negligible accuracy cost.
  This is a deployable recommendation.


## Cell 6: Table 13

In [ ]:
table13 = pd.DataFrame(ens_results)[
    ["Variant", "Accuracy", "Macro F1", "Weighted F1", "MCC"]
].merge(ens_fair, on="Variant", how="left", suffixes=("", "_dup"))
table13 = table13.loc[:, ~table13.columns.str.endswith("_dup")]

base_aod = table13[table13["Variant"] == "Equal-weighted"]["Mean AOD"].values
if len(base_aod):
    table13["AOD vs Equal"] = table13["Mean AOD"] - base_aod[0]
    table13["Improves Fairness?"] = np.where(
        table13["Variant"] == "Equal-weighted", "reference",
        np.where(table13["AOD vs Equal"] < 0, "Yes", "No"))

print("="*100)
print("TABLE 13 — ENSEMBLE WEIGHTING STRATEGY COMPARISON")
print("="*100)
print(table13.round(4).to_string(index=False))

save_result_table(table13, "Table13_Ensemble_Weighting")

print("\nNovelty 7 complete.")
print("\n" + "="*70)
print("ALL 24 EXPERIMENTS AND 7 NOVELTY POINTS FINISHED")
print("="*70)
print("\nRerun Notebook 9 Cell 8 to refresh Table 6 with the final")
print("Explainability Risk and Generalisation Risk scores.")

TABLE 13 — ENSEMBLE WEIGHTING STRATEGY COMPARISON
             Variant  Accuracy  Macro F1  Weighted F1    MCC  Mean AOD  emoji-heavy AOD  slang-heavy AOD  sarcasm AOD    WGA Worst Subgroup  AOD vs Equal Improves Fairness?
      Equal-weighted    0.5952    0.5516       0.5681 0.3447    0.0272          -0.0191           0.0156       0.0468 0.5848    emoji-heavy        0.0000          reference
Performance-weighted    0.5959    0.5585       0.5741 0.3461    0.0307          -0.0160           0.0208       0.0552 0.5869         formal        0.0035                 No
   Fairness-weighted    0.5935    0.5622       0.5769 0.3406    0.0123          -0.0168           0.0167      -0.0034 0.5851         formal       -0.0149                Yes
  saved table -> Table13_Ensemble_Weighting.csv

Novelty 7 complete.

ALL 24 EXPERIMENTS AND 7 NOVELTY POINTS FINISHED

Rerun Notebook 9 Cell 8 to refresh Table 6 with the final
Explainability Risk and Generalisation Risk scores.
